# Fine-tuning Parler-TTS

## Goal of this notebook

In the following notebook, we'll fine-tune [Parler-TTS Mini v0.1](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on a 5h subset of the [Jenny TTS dataset](https://github.com/dioco-group/jenny-tts-dataset), a 30 hours high-quality mono-speaker TTS dataset, from an Irish female speaker named Jenny.

In particular, we'll:
- Annotate the Jenny dataset with natural language speech description using [Data-Speech](https://github.com/huggingface/dataspeech).
- Fine-tune Parler-TTS with the created dataset.

**You should be able to adapt this notebook to your own datasets quite easily.**





## Prepare the Environment

Throughout this tutorial, we'll use a GPU. The runtime is already configured to use the free 16GB T4 GPU provided through Google Colab Free Tier, so all you need to do is hit "Connect T4" in the top right-hand corner of the screen.

##### <a name="installation"> We'll install Parler-TTS and Data-Speech from source in order to train our model.

In [ ]:
# !git clone https://github.com/huggingface/dataspeech.git
!cd dataspeech
# !pip install --quiet -r ./dataspeech/requirements.txt

In [ ]:
!pwd

In [ ]:
# !git clone https://github.com/huggingface/parler-tts.git
%cd parler-tts
!pip install --quiet -e .[train]

Cloning into 'parler-tts'...
remote: Enumerating objects: 1100, done.
remote: Counting objects: 100% (456/456), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1100 (delta 342), reused 278 (delta 278), pack-reused 644 (from 2)
Receiving objects: 100% (1100/1100), 358.00 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (702/702), done.
/home/eblict/parler-tts-finetune/notebook_parler_tts_finetune/parler-tts


On Colab, we need to run an additional set-up, that you can skip if you're on your local machine.

In [ ]:
!pip install --upgrade protobuf wandb==0.16.6

You should link you Hugging Face account so that you can push model repositories on the Hub. This will allow you to save your trained models on the Hub so that you can share them with the community.

Run the command below and then enter an authentication token from https://huggingface.co/settings/tokens. Create a new token if you do not have one already. You should make sure that this token has "write" privileges.

In [ ]:
# !git config --global credential.helper store
# !huggingface-cli login
!hf auth whoami

## 1. Creating our fine-tuning dataset


The aim here is to create an annotated version of Jenny TTS, in order to fine-tune the [Parler-TTS v0.1 checkpoint](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on this dataset.

Thanks to a [script similar to what's described in the Data-Speech FAQ](https://github.com/huggingface/dataspeech?tab=readme-ov-file#how-do-i-use-datasets-that-i-have-with-this-repository), we've uploaded the dataset to the HuggingFace hub, under the name [reach-vb/jenny_tts_dataset](https://huggingface.co/datasets/reach-vb/jenny_tts_dataset).

The purpose of this notebook is demonstration so we've pushed a 6h subset of the dataset that we'll work with: [ylacombe/jenny-tts-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-6h).

Feel free to follow the link above to listen to some samples of the Jenny TTS dataset thanks to the hub viewer.

> Refer to the [Data-Speech README](https://github.com/huggingface/dataspeech?tab=readme-ov-file#data-speech) for more detailed explanations of what's going on under-the-hood.

We'll:
1. Annotate the Jenny dataset with continuous variables that measures the speech characteristics
2. Map those annotations to text bins that characterize the speech characteristics.
3. Create natural language descriptions from those text bins

In [ ]:
%cd ../dataspeech

But first, let's look at a few samples from the Jenny dataset!

In [ ]:
from datasets import load_dataset
dataset = load_dataset("sazzad-sit/kanak30-tts")

In [ ]:
from IPython.display import Audio
print(dataset["train"][0]["text"])
Audio(dataset["train"][0]["audio"]["array"], rate=dataset["train"][0]["audio"]["sampling_rate"])

In [ ]:
from IPython.display import Audio
print(dataset["train"][1]["text"])
Audio(dataset["train"][1]["audio"]["array"], rate=dataset["train"][1]["audio"]["sampling_rate"])

In [ ]:
del dataset


### Annotating the Jenny dataset

We'll use [`main.py`](https://github.com/huggingface/dataspeech/blob/main/main.py) to get the following continuous variables:
- Speaking rate `(nb_phonemes / utterance_length)`
- Signal-to-noise ratio (SNR)
- Reverberation
- Speech monotony


In [1]:
!pwd
# !cd ../dataspeech
%cd /home/eblict/parler-tts-finetune/notebook_parler_tts_finetune/dataspeech
!pwd

/home/eblict/parler-tts-finetune/notebook_parler_tts_finetune
/home/eblict/parler-tts-finetune/notebook_parler_tts_finetune/dataspeech
/home/eblict/parler-tts-finetune/notebook_parler_tts_finetune/dataspeech


In [5]:
!python main.py "sazzad-sit/kanak30-tts" \
  --text_column_name "text" \
  --audio_column_name "audio" \
  --cpu_num_workers 2 \
  --num_workers_per_gpu_for_pitch 2 \
  --rename_column \
  --repo_id "kanak30-tts-tags"

Initializing phonemizer...
/home/eblict/miniconda3/envs/ptts/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_verification.py:43: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import (
Compute pitch
Compute snr and reverb
Compute speaking rate
Pushing to the hub...
Creating parquet from Arrow format: 100%|███████| 18/18 [00:00<00:00, 83.05ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.

Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|███████████████| 1/1 [00:01<00:00,  1.15s/it]
No files have been modified since last commit. Skipping to prevent empty commit.


The whole process took under 10mn!

The resulting dataset will be pushed to the HuggingFace hub under your HuggingFace handle. Mine was push to [ylacombe/jenny-tts-tags-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-tags-6h).

Let's see what the new dataset looks like:

In [ ]:
from datasets import load_dataset
dataset = load_dataset("sazzad-sit/kanak30-tts-tags")
print("SNR 1st sample", dataset["train"][0]["snr"])
print("C50 2nd sample", dataset["train"][0]["c50"])
del dataset

As you can see, the current annotations are continuous variables. To use it with Parler-TTS, we need to convert it to textual description, something that the two next steps will take care of.

### 2. Map annotations to text bins

Since the ultimate goal here is to fine-tune the [Parler-TTS v0.1 checkpoint](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on the Jenny dataset, we want to stay consistent with the text bins of the datasets on which the latter model was trained.

This is easy to do thanks to the following:

In [ ]:
!python ./scripts/metadata_to_text.py \
    "sazzad-sit/kanak30-tts-tags" \
    --repo_id "kanak30-tts-tags" \
    --configuration "default" \
    --cpu_num_workers 2 \
    --path_to_bin_edges "./examples/tags_to_annotations/v01_bin_edges.json" \
    --avoid_pitch_computation

Thanks to [`v01_bin_edges.json`](https://github.com/huggingface/dataspeech/blob/main/examples/tags_to_annotations/v01_bin_edges.json), we don't need to recompute bins from scratch and the above script takes a few seconds.

The resulting dataset will be pushed to the HuggingFace hub under your HuggingFace handle. Mine was push to [ylacombe/jenny-tts-tags-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-tags-6h).

You can notice that text bins such as `quite noisy`, `very fast` have been added to the samples.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("sazzad-sit/kanak30-tts-tags")
print("Noise 1st sample:", dataset["train"][0]["noise"])
print("Speaking rate 2nd sample:", dataset["train"][0]["speaking_rate"])
del dataset


### 3. Create natural language descriptions from those text bins

Now that we have text bins associated to the Jenny dataset, the next step is to create natural language descriptions out of the few created features.

Here, we decided to create prompts that use the name `Jenny`, prompts that'll look like the following:
`In a very expressive voice, Jenny pronounces her words incredibly slowly. There's some background noise in this room with a bit of echo'`

This step generally demands more resources and times and should use one or many GPUs.

The following command shows how to do it using the [2B version of the Gemma model from Google](https://huggingface.co/google/gemma-2b-it), which should run in about 15 minutes in this Colab free T4.


As usual, we precise the dataset name and configuration we want to annotate. `model_name_or_path` should point to a `transformers` model for prompt annotation. You can find a list of such models [here](https://huggingface.co/models?pipeline_tag=text-generation&library=transformers&sort=trending).

**Note** how we've been able to specify that the dataset is mono-speaker and that we should name the voice Jenny thanks to the flags:


`--speaker_name "Jenny" --is_single_speaker`.


In [ ]:
!python ./scripts/run_prompt_creation.py \
  --speaker_name "Kanak" \
  --is_single_speaker \
  --dataset_name "sazzad-sit/kanak30-tts-tags" \
  --output_dir "./tmp_kanak" \
  --dataset_config_name "default" \
  --model_name_or_path "google/gemma-2b-it" \
  --per_device_eval_batch_size 12 \
  --attn_implementation "sdpa" \
  --dataloader_num_workers 2 \
  --push_to_hub \
  --hub_dataset_id "kanak30-tts-tagged" \
  --preprocessing_num_workers 2

Let's take a look at some created prompts:

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ylacombe/jenny-tts-6h-tagged")
print("1st sample:", dataset["train"][0]["text_description"])
print("2nd sample:", dataset["train"][1]["text_description"])
del dataset

**Observation:** The first sample unfortunately doesn't have the name Jenny in it. This is probably because we use a smaller and thus less precise model that one we would have gone for if this notebook had more resources (e.g we've used [Mistral 7B v2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) to create the Parler-TTS training dataset). This shouldn't prevent our model to learn what we want though.

## Fine-tuning Parler-TTS



In [ ]:
%cd ../parler-tts

We can know fully focus on fine-tuning Parler-TTS. Luckily, [the Parler-TTS library](https://github.com/huggingface/.parler-tts) has a training script available [here](https://github.com/huggingface/parler-tts/tree/main/training), that can be used with just a few arguments.


> **Note:** you need to enter your choice concerning WandB. If you don't have an account, you can enter `3` to avoid logging on WandB. Otherwise; you can logging to follow how your model trained.

In [ ]:
# !accelerate launch --mixed_precision fp16 ./training/run_parler_tts_training.py \
#     --model_name_or_path "ai4bharat/indic-parler-tts" \
#     --feature_extractor_name "ylacombe/dac_44khz" \
#     --description_tokenizer_name "google/flan-t5-large" \
#     --prompt_tokenizer_name "ai4bharat/indic-parler-tts" \
#     --report_to "none" \
#     --overwrite_output_dir true \
#     --train_dataset_name "sazzad-sit/kanak30-tts" \
#     --train_metadata_dataset_name "sazzad-sit/kanak30-tts-tagged" \
#     --train_dataset_config_name "default" \
#     --train_split_name "train" \
#     --eval_dataset_name "sazzad-sit/kanak30-tts" \
#     --eval_metadata_dataset_name "sazzad-sit/kanak30-tts-tagged" \
#     --eval_dataset_config_name "default" \
#     --eval_split_name "train" \
#     --target_audio_column_name "audio" \
#     --description_column_name "text_description" \
#     --prompt_column_name "text" \
#     --max_duration_in_seconds 20 \
#     --min_duration_in_seconds 2.0 \
#     --max_text_length 400 \
#     --preprocessing_num_workers 2 \
#     --do_train true \
#     --do_eval true \
#     --num_train_epochs 2 \
#     --gradient_accumulation_steps 18 \
#     --gradient_checkpointing true \
#     --per_device_train_batch_size 2 \
#     --learning_rate 0.00008 \
#     --adam_beta1 0.9 \
#     --adam_beta2 0.99 \
#     --weight_decay 0.01 \
#     --lr_scheduler_type "constant_with_warmup" \
#     --warmup_steps 50 \
#     --logging_steps 2 \
#     --freeze_text_encoder true \
#     --audio_encoder_per_device_batch_size 4 \
#     --dtype "float16" \
#     --seed 456 \
#     --output_dir "./output_dir_kanak_finetune/" \
#     --temporary_save_to_disk "./audio_code_tmp/" \
#     --save_to_disk "./tmp_dataset_audio/" \
#     --dataloader_num_workers 2 \
#     --predict_with_generate \
#     --include_for_metrics "inputs" \
#     --group_by_length true \
#     --attn_implementation "flash_attention_2"


!accelerate launch ./training/run_parler_tts_training.py \
    --model_name_or_path "ai4bharat/indic-parler-tts" \
    --feature_extractor_name "ylacombe/dac_44khz" \
    --description_tokenizer_name "google/flan-t5-large" \
    --prompt_tokenizer_name "ai4bharat/indic-parler-tts" \
    --report_to "none" \
    --overwrite_output_dir true \
    --train_dataset_name "sazzad-sit/kanak30-tts" \
    --train_metadata_dataset_name "sazzad-sit/kanak30-tts-tagged" \
    --train_dataset_config_name "default" \
    --train_split_name "train" \
    --eval_dataset_name "sazzad-sit/kanak30-tts" \
    --eval_metadata_dataset_name "sazzad-sit/kanak30-tts-tagged" \
    --eval_dataset_config_name "default" \
    --eval_split_name "train" \
    --max_eval_samples 8 \
    --per_device_eval_batch_size 8 \
    --target_audio_column_name "audio" \
    --description_column_name "text_description" \
    --prompt_column_name "text" \
    --max_duration_in_seconds 20 \
    --min_duration_in_seconds 2.0 \
    --max_text_length 400 \
    --preprocessing_num_workers 2 \
    --do_train true \
    --num_train_epochs 2 \
    --gradient_accumulation_steps 18 \
    --gradient_checkpointing true \
    --per_device_train_batch_size 2 \
    --learning_rate 0.00008 \
    --adam_beta1 0.9 \
    --adam_beta2 0.99 \
    --weight_decay 0.01 \
    --lr_scheduler_type "constant_with_warmup" \
    --warmup_steps 50 \
    --logging_steps 2 \
    --freeze_text_encoder true \
    --audio_encoder_per_device_batch_size 4 \
    --dtype "float16" \
    --seed 456 \
    --output_dir "./output_dir_kanak_finetune/" \
    --temporary_save_to_disk "./audio_code_tmp/" \
    --save_to_disk "./tmp_dataset_audio/" \
    --dataloader_num_workers 2 \
    --do_eval \
    --predict_with_generate \
    --include_inputs_for_metrics \
    --group_by_length true

In [ ]:
%cd ./parler-tts/

## Inference

The full training on the free T4 from Google Colab took about an hour.
Now, let's see how to do inference with the newly fine-tuned model!

First install the Parler-TTS library:

In [ ]:
!pip install git+https://github.com/huggingface/parler-tts.git

Then:

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained("/content/parler-tts/output_dir_training", torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler_tts_mini_v0.1")

prompt = "Hey, how are you doing today?"
description = "'Jenny delivers her words quite expressively, in a very confined sounding environment with clear audio quality. She speaks fast.'"

input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

In [ ]:
from IPython.display import Audio
Audio(audio_arr, rate=model.config.sampling_rate)

In [ ]:
prompt = "Wow, I've really got the same voice as Jenny, huh?"

prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

Audio(audio_arr, rate=model.config.sampling_rate)

In [ ]:
prompt = "What a time to be alive!"
description = "'Jenny's speech is very clear, and she speaks in a very monotone voice, really slowly and with minimal variation in speed.'"

input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

Audio(audio_arr, rate=model.config.sampling_rate)

This is great! As you can see, the model now managed to get a **consistent** voice throughout generation that looks like **Jenny**!

Since we're quite happy about it, let's push it to the hub to be able to re-use it!

In [ ]:
model.push_to_hub("parler-tts-mini-Jenny-colab")
tokenizer.push_to_hub("parler-tts-mini-Jenny-colab")

You'll now be able to load the model and the tokenizer using the direct repository id of your model, i.e `<your_HF_handle>/parler-tts-mini-Jenny-colab`.

```python
model = ParlerTTSForConditionalGeneration.from_pretrained("<your_HF_handle>/parler-tts-mini-Jenny-colab").to(device)
tokenizer = AutoTokenizer.from_pretrained("<your_HF_handle>/parler-tts-mini-Jenny-colab")
```



## Conclusion

To conclude, we've shown here:
1. how to annotate a single-speaker 6-hours-long dataset
2. how to fine-tune Parler-TTS Mini v0.1 on this newly created dataset!

**If you want to fine-tune the model on your own dataset, you can follow and/or adapt the current notebook to make it work! Don't forget to check how to push your own local dataset on the HuggingFace Hub using a [script similar to what's described in the Data-Speech FAQ](https://github.com/huggingface/dataspeech?tab=readme-ov-file#how-do-i-use-datasets-that-i-have-with-this-repository)!**